## Загрузка датасета SOBHard на HuggingFace

Ноутбук собирает `test.json` и `shots.json` из `datasets/SOBHard/` в `datasets.DatasetDict`
и заливает его на 🤗 Hub по указанному пути.

Что здесь специфично для SOBHard:
* поле `instruction` в локальных файлах — это **индекс** промпта в `dataset_meta.json["prompts"]`;
  перед заливкой он заменяется на сам текст промпта (общее правило MERA);
* `meta.reference` — эталонный документ, по которому считаются метрики содержания. Скоринг читает
  эталон именно оттуда, а не из `outputs`, поэтому это поле обязано доехать до Hub без изменений,
  если вы хотите получать метрики `content_pass_rate` и `sample_pass_rate`;
* `meta.checks` — JSON-строка с предвычисленными данными для проверок, выводимых из исходного
  документа; без неё три проверки просто не применяются;
* перед заливкой прогоняется проверка целостности: каждый эталонный ответ обязан пройти все
  применимые к нему ограничения тем же скорером, которым считается метрика.


In [ ]:
import json
import os
import sys

import datasets
from tqdm import tqdm


### Подготовка данных


#### WARNING!

Если ваш датасет является __ПРИВАТНЫМ__, оставьте `MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS` равным `True`.
Иначе поставьте `False`. Этот флаг дальше используется, чтобы стереть ответы перед загрузкой на ХФ.
На ХФ даже приватно не должно лежать датасетов с ответами!

**Важное отличие SOBHard от IFHardBench.** В IFHardBench скоринг не смотрит на эталон вовсе, поэтому
стирание `outputs` там ничего не ломает. Здесь задание состоит в том, чтобы выдать конкретный документ,
и без эталона метрики содержания посчитать нечем. Эталон лежит в `meta.reference`, и стирание одного лишь
`outputs` ответы **не прячет**.

Поэтому решение придётся принять явно, флагом `KEEP_REFERENCE_FOR_SCORING` ниже. Ячейка «Что мы теряем»
покажет на числах, какие метрики выживут при каждом варианте, — решайте по ней, а не наугад.


In [ ]:
MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS = True

# True  — meta.reference остаётся на Hub. Считаются все пять метрик, но эталонные
#         документы фактически опубликованы (пусть и в приватном репозитории).
# False — meta.reference стирается вместе с outputs. Датасет не содержит ответов,
#         но content_pass_rate и sample_pass_rate становятся неинформативными.
KEEP_REFERENCE_FOR_SCORING = True


Параметр `path_to_data` — путь ДО файлов `shots.json` и `test.json`.

Параметр `path_to_meta` — путь ДО `dataset_meta.json`.

Пути ниже указаны относительно расположения ноутбука в `datasets/SOBHard/`; поменяйте, если запускаете из другого места.


In [ ]:
path_to_data = "."
path_to_meta = "."


Сплиты и мета лежат в формате JSON.


In [ ]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data


#### Подгрузка данных


In [ ]:
shots = load_json(os.path.join(path_to_data, "shots.json"))["data"]
test = load_json(os.path.join(path_to_data, "test.json"))["data"]
meta = load_json(os.path.join(path_to_meta, "dataset_meta.json"))

print(f"shots: {len(shots)}, test: {len(test)}")


Из меты для датасета нужны только промпты.


In [ ]:
prompts = meta["prompts"]
len(prompts)


#### Обработка полей датасета

На ХФ загружается датасет, где у КАЖДОГО сэмпла вместо числа в поле `instruction` стоит промпт.
Число указывает, какой по индексу взять промпт из секции с промптами в мете датасета.

Ячейка идемпотентна: если её случайно выполнить дважды, строки не будут перезаписаны повторно.


In [ ]:
def resolve_prompts(split):
    for card in split:
        if isinstance(card["instruction"], int):
            card["instruction"] = prompts[card["instruction"]]


resolve_prompts(shots)
resolve_prompts(test)

print(test[0]["instruction"][:400])


#### Проверка целостности перед заливкой

Подставляем `inputs` в `instruction` — промпт должен собираться без ошибок — и прогоняем эталонный
ответ через ту же библиотеку проверок, которой считается метрика. Эталон обязан набрать 1.0 по всем
метрикам: скорер, заваливающий идеальный ответ, не измеряет ничего. Если что-то не сходится,
на Hub такой датасет заливать нельзя.


In [ ]:
sys.path.insert(0, os.path.abspath("../../benchmark_tasks/sobhard"))
import utils as U

bad_prompt, bad_gold = [], []
for card in tqdm(shots + test):
    try:
        card["instruction"].format(**card["inputs"])
    except Exception as exc:
        bad_prompt.append((card["meta"]["id"], exc))
    m = U.process_results(card, [card["outputs"]])
    if any(v != 1.0 for v in m.values()):
        bad_gold.append((card["meta"]["id"], m))

print("промптов, которые не собираются:", len(bad_prompt))
print("эталонов, не набравших 1.0:", len(bad_gold))
assert not bad_prompt and not bad_gold


#### Что мы теряем, если стереть эталон

Ячейка ничего не меняет: она берёт копию данных, стирает в ней `meta.reference` и показывает,
сколько проверок остаётся применимыми. Это и есть цена варианта `KEEP_REFERENCE_FOR_SCORING = False`.


In [ ]:
import copy
from collections import Counter

probe = copy.deepcopy(test[:50])
for card in probe:
    card["meta"]["reference"] = ""
    card["outputs"] = ""

alive = Counter()
for card in probe:
    for ch in U.score_response(card, "```json\n{}\n```"):
        if ch["applicable"]:
            alive[ch["id"]] += 1

print("проверок остаётся применимыми (на выборке из 50 вопросов):")
for cid in U.constraint_ids():
    print(f"  {cid:32s} {alive.get(cid, 0):>3d}")
print()
print("без эталона теряются все val.* и две проверки transform;")
print("format_pass_rate и часть task_pass_rate продолжают считаться.")


#### Убираем ответы для приватных задач

Надеемся, вы поставили в начале ноутбука корректные значения обоих флагов.

В `test` стираются ответы; в `shots` они сохраняются — это few-shot примеры, они и должны быть видны модели.


In [ ]:
def hide_answers(dataset_split: list, drop_reference: bool):
    for card in tqdm(dataset_split):
        card["outputs"] = ""
        if drop_reference:
            card["meta"]["reference"] = ""


In [ ]:
if MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS:
    hide_answers(test, drop_reference=not KEEP_REFERENCE_FOR_SCORING)

print("outputs стёрт:", all(c["outputs"] == "" for c in test)
      if MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS else "outputs оставлен")
print("meta.reference на месте:", all(c["meta"]["reference"] for c in test))


### Создаем датасет для загрузки на ХФ


#### Аннотация полей датасета

В `features` повторяется структура КАЖДОГО сэмпла датасета с описанием формата данных в каждом поле.

Обратите внимание на места, специфичные для SOBHard:
* `meta.task_meta` и `meta.checks` — именно **строки** с JSON внутри, а не вложенные структуры:
  у разных семейств разный набор параметров (`{"extract_paths": [...]}`, `{"break_kind": "trailing_comma"}`,
  `{"transform": "sort_keys_recursive"}`), и фиксированной схемой их не описать;
* `meta.reference` — сырой текст эталонного документа, а не разобранный JSON. Хранить его разобранным нельзя:
  в YAML и TOML ключи отображений бывают не строками, и JSON-сериализация превратила бы их в строки
  только с одной стороны сравнения.


In [ ]:
features = datasets.Features({
    "instruction": datasets.Value("string"),
    "inputs": {
        "task": datasets.Value("string"),
        "input_data": datasets.Value("string"),
        "format": datasets.Value("string"),
        "question": datasets.Value("string"),
    },
    "outputs": datasets.Value("string"),
    "meta": {
        "id": datasets.Value("int32"),
        "base_id": datasets.Value("string"),
        "fence_tag": datasets.Value("string"),
        "reference": datasets.Value("string"),
        "reference_sha256": datasets.Value("string"),
        "task_meta": datasets.Value("string"),
        "checks": datasets.Value("string"),
        "categories": {
            "family": datasets.Value("string"),
            "difficulty": datasets.Value("string"),
            "language": datasets.Value("string"),
            "source_format": datasets.Value("string"),
            "target_format": datasets.Value("string"),
            "length_tier": datasets.Value("string"),
            "prompt_style": datasets.Value("string"),
            # Provenance: which corpus file the document came from, and which
            # subtree of it. Kept so a question can be traced back to real data.
            "origin": datasets.Value("string"),
        },
    },
})


#### Создание датасетов для каждого сплита


In [ ]:
shots_ds = datasets.Dataset.from_list(shots, features=features)
test_ds = datasets.Dataset.from_list(test, features=features)
shots_ds, test_ds


##### Проверка

Проверим, что сборка прошла успешно — ничего не потеряно, не продублировано и не переехало.


In [ ]:
# количество вопросов до конвертации и после совпадает
assert len(test) == len(test_ds) and len(shots) == len(shots_ds)

# id вопросов сходятся и остаются сквозными
assert [c["meta"]["id"] for c in test] == [c["meta"]["id"] for c in test_ds]
assert [c["meta"]["id"] for c in shots] == [c["meta"]["id"] for c in shots_ds]
ids = sorted(c["meta"]["id"] for c in shots + test)
assert ids == list(range(1, len(ids) + 1))

# сетка 25 вопросов в каждой из 20 клеток пережила конвертацию
from collections import Counter
cells = Counter((c["meta"]["categories"]["family"], c["meta"]["categories"]["difficulty"])
                for c in test_ds)
assert len(cells) == 20 and set(cells.values()) == {25}, cells

# meta.checks и meta.task_meta всё ещё парсятся
assert all(json.loads(c["meta"]["checks"]) is not None for c in test_ds)
assert all(json.loads(c["meta"]["task_meta"]) is not None for c in test_ds)
print("OK")


#### Собираем сплиты в один датасет


In [ ]:
dataset = datasets.DatasetDict({"shots": shots_ds, "test": test_ds})
dataset


### Загрузка датасета на ХФ

Для загрузки на ХФ понадобятся:
- Токен — строка с ключом, дающим право записи в репозиторий.
- Путь для записи — аккаунт и название датасета. Название пишите ровно так, как оно заявлено в мете
  (`dataset_meta.json["dataset_name"]`), регистр имеет значение.

Советуем сначала залить всё приватно и выслать на почту mera@a-ai.ru токен и путь для верификации.


In [ ]:
from dotenv import load_dotenv

# Загружаем переменные из .env файла
load_dotenv('../../.env')

### TOKEN
token = os.getenv('HF_TOKEN')
if token is None:
    raise ValueError("HF_TOKEN not found in .env file")

### UPLOAD PATH — поменяйте на нужный вам путь
HF_REPO_ID = "MERA-evaluation/SOBHard"

# Чтобы предварительно посмотреть, как датасет будет выглядеть после заливки,
# можно сначала загрузить его в свой приватный репозиторий:
# HF_REPO_ID = "<your-account>/SOBHard"

### PRIVATE OR PUBLIC
upload_private = True

print("upload to:", HF_REPO_ID, "| private:", upload_private)


In [ ]:
dataset.push_to_hub(HF_REPO_ID, private=upload_private, token=token)


После заливки не забудьте поменять `dataset_path` в `benchmark_tasks/sobhard/sobhard.yaml`,
если путь отличается от `MERA-evaluation/SOBHard`, и переключиться с задачи `sobhard_local` на `sobhard`.


### Проверка того, как датасет загрузился на ХФ

Загрузим датасет обратно и убедимся, что его увидит корректно любой, кто его скачает:
все поля на месте, содержание совпадает с исходным, а промпт по-прежнему собирается.


In [ ]:
ds = datasets.load_dataset(HF_REPO_ID, token=token)
ds


In [ ]:
check = []
for idx, card in enumerate(ds["test"]):
    same_question = test[idx]["inputs"]["question"] == card["inputs"]["question"]
    same_reference = test[idx]["meta"]["reference"] == card["meta"]["reference"]
    same_checks = test[idx]["meta"]["checks"] == card["meta"]["checks"]
    check.append(same_question and same_reference and same_checks)

all(check)


In [ ]:
# промпт собирается из скачанной с Hub копии — ровно то, что будет делать lm-eval
card = ds["test"][0]
print(card["instruction"].format(**card["inputs"])[:600])


#### Финальная проверка: метрики считаются по скачанной копии

Если `KEEP_REFERENCE_FOR_SCORING = True`, эталон из `meta.reference` должен набирать 1.0 по всем метрикам
прямо на данных с Hub. Это последняя точка, где ошибку ещё видно до запуска модели.


In [ ]:
sample = ds["test"][0]
gold = f"```{sample['meta']['fence_tag']}\n{sample['meta']['reference']}\n```"
print(U.process_results(sample, [gold]))
